<!-- NB_DOC_INTRO_V1 -->
# Time Series — ARIMA, SARIMA, ARIMAX (en profondeur)

> 📚 **Doc thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Time Series & Traitement du Signal)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Focus complet ARIMA** : AR (auto-regressive), MA (moving average), I (integrated = differentiation), S (seasonal), X (exogenous). Notation `ARIMA(p, d, q)` et `SARIMA(p, d, q)(P, D, Q, s)`. Ce notebook execute le **workflow Box-Jenkins** : identification (ACF/PACF), estimation, diagnostic, forecast.

Auto-ARIMA via `pmdarima` (ou statsmodels en fallback).

## Plan

1. Vocabulaire ARIMA (p, d, q, P, D, Q, s)
2. Workflow Box-Jenkins
3. ACF / PACF → identification p, q
4. Auto-ARIMA (pmdarima)
5. SARIMA detaille
6. ARIMAX (avec variables exogenes)
7. Diagnostic des residus
8. Forecast + intervals
9. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Vocabulaire ARIMA

`SARIMA(p, d, q)(P, D, Q, s)` :
- **AR(p)** : `y_t = c + φ_1·y_{t-1} + ... + φ_p·y_{t-p} + ε_t` (auto-regression d'ordre p)
- **MA(q)** : `y_t = μ + ε_t + θ_1·ε_{t-1} + ... + θ_q·ε_{t-q}` (moyenne mobile sur les erreurs)
- **I(d)** : `d` differentiations pour rendre stationnaire
- **S** : ajoute saisonnier avec periode `s` (ex: 12 pour mensuel annuel)
- **X** : variables exogenes additionnelles

| Cas | Notation |
|---|---|
| Trend lineaire stationnaire | ARIMA(p, 0, q) |
| Avec tendance, sans saison | ARIMA(p, 1, q) |
| Annuel saisonnier (mensuel) | SARIMA(p, d, q)(P, D, Q, 12) |
| Avec features externes | ARIMAX(p, d, q, exog=X) |


## 2. Dataset + analyse

In [ ]:
dates = pd.date_range("2000-01-01", periods=120, freq="MS")
trend = np.linspace(100, 500, 120)
season = 60 * np.sin(2 * np.pi * np.arange(120) / 12)
noise = np.random.normal(0, 15, 120)
ts = pd.Series(trend + season + noise, index=dates)

split = int(len(ts) * 0.8)
train, test = ts[:split], ts[split:]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train.values, label="Train")
ax.plot(test.index, test.values, label="Test", color="C3")
ax.set_title("Series TS"); ax.legend()
plt.show()

## 3. Stationnarisation

In [ ]:
def adf_test(s, name=""):
    res = adfuller(s.dropna())
    print(f"{name} ADF stat={res[0]:.3f}, p={res[1]:.4f} → {'STATIONNAIRE' if res[1] < 0.05 else 'non-stationnaire'}")

adf_test(train, "Original")
adf_test(train.diff(), "Diff(1)")
adf_test(train.diff().diff(12), "Diff(1).Diff(12)")

## 4. ACF / PACF → identification p, q

In [ ]:
# Apres differentiation suffisante (d=1, D=1)
ts_stat = train.diff().diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(ts_stat, lags=30, ax=axes[0])
axes[0].set_title("ACF (suggere q)")
plot_pacf(ts_stat, lags=30, ax=axes[1])
axes[1].set_title("PACF (suggere p)")
plt.tight_layout(); plt.show()

**Lecture** :
- PACF coupe apres lag p (significatif jusqu'a lag p, ~ 0 ensuite) → AR(p)
- ACF coupe apres lag q → MA(q)
- Decroissance lente sans cut → besoin de + de differentiation

## 5. Auto-ARIMA (pmdarima)


In [ ]:
# pmdarima auto-select (lent sur gros datasets, mais commode)
try:
    import pmdarima as pm
    auto = pm.auto_arima(train, seasonal=True, m=12,
                          start_p=0, max_p=3, start_q=0, max_q=3,
                          start_P=0, max_P=2, start_Q=0, max_Q=2,
                          d=1, D=1, trace=False, suppress_warnings=True,
                          error_action="ignore", stepwise=True)
    print(auto.summary())
    AUTO_ORDER = auto.order
    AUTO_SEAS = auto.seasonal_order
    print(f"\nBest ARIMA : {AUTO_ORDER}({AUTO_SEAS})")
except ImportError:
    print("pmdarima pas installe — fallback manuel")
    AUTO_ORDER = (1, 1, 1)
    AUTO_SEAS = (1, 1, 1, 12)

## 6. SARIMA — fit + forecast

In [ ]:
sarima = SARIMAX(train, order=AUTO_ORDER, seasonal_order=AUTO_SEAS,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(sarima.summary().tables[1])

forecast = sarima.get_forecast(steps=len(test))
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int(alpha=0.05)

print(f"\nRMSE  : {root_mean_squared_error(test, fc_mean):.2f}")
print(f"MAE   : {mean_absolute_error(test, fc_mean):.2f}")

## 7. Visualisation forecast + intervalles

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train.index, train.values, label="Train")
ax.plot(test.index, test.values, label="Test (vrai)", color="black", linewidth=2)
ax.plot(fc_mean.index, fc_mean.values, label="Forecast", color="red")
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1], color="red", alpha=0.2, label="95% CI")
ax.legend(); ax.set_title(f"SARIMA{AUTO_ORDER}{AUTO_SEAS}")
plt.show()

## 8. ARIMAX avec exogenous variables

In [ ]:
# Exogene jouet : un evenement marketing 1 mois sur 6
exog_train = pd.DataFrame({"marketing": (np.arange(len(train)) % 6 == 0).astype(int)}, index=train.index)
exog_test = pd.DataFrame({"marketing": (np.arange(len(train), len(train)+len(test)) % 6 == 0).astype(int)}, index=test.index)

# Note : pour effets reels, l'exog doit avoir un VRAI lien causal avec y
arimax = SARIMAX(train, exog=exog_train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
arimax_forecast = arimax.get_forecast(steps=len(test), exog=exog_test).predicted_mean
print(f"ARIMAX — RMSE : {root_mean_squared_error(test, arimax_forecast):.2f}")

## 9. Diagnostic des residus

Un bon modele ARIMA a des residus = **bruit blanc** : pas autocorreles, distribution normale.

In [ ]:
residuals = sarima.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
residuals.plot(ax=axes[0, 0], title="Residuals over time")
axes[0, 0].axhline(0, color="red", linestyle="--")

axes[0, 1].hist(residuals, bins=20, edgecolor="k")
axes[0, 1].set_title("Histogram (devrait etre normal)")

from statsmodels.graphics.gofplots import qqplot
qqplot(residuals, line="s", ax=axes[1, 0])
axes[1, 0].set_title("Q-Q plot")

plot_acf(residuals, lags=20, ax=axes[1, 1])
axes[1, 1].set_title("ACF residus (devrait etre proche de 0)")

plt.tight_layout(); plt.show()

# Test Ljung-Box (H0 : pas d'autocorrelation residuelle)
lb = acorr_ljungbox(residuals, lags=[10, 20], return_df=True)
print(f"\nLjung-Box test :\n{lb}")
print(f"\np > 0.05 → pas d'autocorr → modele OK")

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Choisir ARIMA(p, d, q) au hasard | Workflow Box-Jenkins : ADF + ACF/PACF + diagnostic |
| Differentiation excessive (d > 2 sans raison) | Sur-differentier introduit du bruit (over-differencing) |
| Pas verifier les residus | Bruit blanc obligatoire pour valider |
| Ignorer la saisonnalite | Si periodicite visible → SARIMA |
| Trop de parametres (p+q grand) | Overfit, AIC/BIC favorise les simples |
| `enforce_stationarity=True` quand convergence echoue | Mettre False (moins strict) |
| Comparer modeles via RMSE sur train | Toujours sur test (out-of-sample) |
| Predire trop loin (horizon long) | Incertitude explose — limiter horizon |


## References

### Documentation
- statsmodels ARIMA : https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html
- pmdarima : https://alkaline-ml.com/pmdarima/
- Hyndman ARIMA chapter : https://otexts.com/fpp3/arima.html

### Voir aussi
- [TS_Time_Series_Intro.ipynb](TS_Time_Series_Intro.ipynb)
- [TS_Time_Series_Overview.ipynb](TS_Time_Series_Overview.ipynb)
- [TS_Maintenance_Predictive_GOOD.ipynb](TS_Maintenance_Predictive_GOOD.ipynb)
